In [1]:
!pip install mediapipe

  Using cached mediapipe-1.0.1-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sounddevice-0.5.6-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached mediapipe-1.0.1-py3-none-win_amd64.whl (20.1 MB)
Using cached absl_py-2.5.0-py3-none-any.whl (137 kB)
Using cached sounddevice-0.5.6-py3-none-win_amd64.whl (1.0 MB)
Using cached opencv_contrib_python-5.0.0.93-cp37-abi3-win_amd64.whl (53.8 MB)

   ---------------------------------------- 0/4 [opencv-contrib-python]
   ---------------------------------------- 0/4 [opencv-contrib-python]
   ---------------------------------------- 0/4 [opencv-contrib-python]
   ---------------------------------------- 0/4 [opencv-contrib-python]
   ---------- ----------------------------- 1/4 [absl-py]
   ------------------------------ --------- 3/4 [mediapipe]
   ------------------------------ -------

In [2]:
pip install screen-brightness-control

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import numpy as np
from math import hypot
import urllib.request
import os
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

C:\Users\Pratik\anaconda3\Lib\site-packages\mediapipe\__init__.py


In [3]:
import urllib.request

url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
urllib.request.urlretrieve(url, "hand_landmarker.task")
print("hand_landmarker.task downloaded successfully!")

hand_landmarker.task downloaded successfully!


In [5]:
model_path = 'hand_landmarker.task'
if not os.path.exists(model_path):
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    print("Downloading hand_landmarker.task...")
    urllib.request.urlretrieve(url, model_path)
    print("Download complete.")

In [6]:
# 2. Initialize HandLandmarker Model
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.7,
    min_tracking_confidence=0.7
)
detector = vision.HandLandmarker.create_from_options(options)

# 21 Hand Landmark Connections
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),           # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),           # Index finger
    (5, 9), (9, 10), (10, 11), (11, 12),      # Middle finger
    (9, 13), (13, 14), (14, 15), (15, 16),    # Ring finger
    (13, 17), (17, 18), (18, 19), (19, 20),   # Pinky
    (0, 17)                                   # Palm base
]


In [7]:
# 3. Start Webcam Stream
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

meter_bar = 400
meter_percent = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    height, width, _ = frame.shape

    # Convert BGR frame to RGB MediaPipe Image
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

    # Detect landmarks
    detection_result = detector.detect(mp_image)

    if detection_result.hand_landmarks:
        for hand in detection_result.hand_landmarks:
            # Convert normalized coordinates (0.0 to 1.0) to pixel coordinates
            pts = [(int(lm.x * width), int(lm.y * height)) for lm in hand]

            # Draw skeleton connections
            for start, end in HAND_CONNECTIONS:
                cv2.line(frame, pts[start], pts[end], (220, 220, 220), 2)

            # Draw landmark joints
            for pt in pts:
                cv2.circle(frame, pt, 4, (0, 0, 255), cv2.FILLED)

            # Landmark 4: Thumb Tip, Landmark 8: Index Tip
            x1, y1 = pts[4]
            x2, y2 = pts[8]
            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            # Draw pinch interaction indicators
            cv2.circle(frame, (x1, y1), 8, (255, 0, 255), cv2.FILLED)
            cv2.circle(frame, (x2, y2), 8, (255, 0, 255), cv2.FILLED)
            cv2.line(frame, (x1, y1), (x2, y2), (255, 0, 255), 3)
            cv2.circle(frame, (cx, cy), 6, (0, 255, 255), cv2.FILLED)

            # Calculate distance between thumb and index tips
            dist = hypot(x2 - x1, y2 - y1)

            # Map distance (approx 25px - 180px) to percentage and UI meter
            meter_percent = int(np.interp(dist, [25, 180], [0, 100]))
            meter_bar = int(np.interp(dist, [25, 180], [400, 150]))

            # Visual feedback on pinch close
            if dist < 30:
                cv2.circle(frame, (cx, cy), 10, (0, 255, 0), cv2.FILLED)

    # Draw On-Screen Gesture Meter
    cv2.rectangle(frame, (50, 150), (85, 400), (0, 200, 255), 3)
    cv2.rectangle(frame, (50, meter_bar), (85, 400), (0, 200, 255), cv2.FILLED)
    cv2.putText(frame, f'{meter_percent}%', (40, 435), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
    cv2.putText(frame, 'Gesture', (35, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 200, 255), 2)

    cv2.imshow('Hand Gesture Tracking', frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()